# Script Outline

SACOG keeps tracks of commute modes. One primary source is American Community Survey 1-Year Summary data.  Please write a script in a language of your choice (Preferred in Python) to download/process the data For SACOG region, including 6 counties (El Dorado, Placer, Sacramento, Sutter, Yolo, and Yuba), through Census API.

- Prepare Workspace
- Prepare Mapping Tables
- Import Data
- Data Cleaning
- Data Checks
- Exports


Sources:

https://api.census.gov/data/2019/acs/acs1/examples.html

https://medium.com/@mcmanus_data_works/using-the-u-s-census-bureau-api-with-python-5c30ad34dbd7

https://pypi.org/project/census/

https://www.census.gov/content/dam/Census/library/publications/2018/acs/acs_general_handbook_2018_ch03.pdf

## Prepare Workspace

#### Import Packages

In [1]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast


# Geographic
import geopandas as gpd
from census import Census
from us import states
import censusdata as acs

#### User defined functions

In [ ]:
def acs1_state_county(api_Key, variables, year, state, county):
    
    '''
    User defined function to import ACS 1 year estimates at the county level
    Fixed inputs: [host_, dataset_, g_] to construct URL
    User inputs: [api_key_, variables_, year_, location_] to tell ACS that we have access with the API key and
                    to tell ACS which variables we want to import, what year, and which state and county
                    (currently can only do 1 state and county at a time)
    '''
    
    # Fixed inputs
    host_ = 'https://api.census.gov/data'
    dataset_ = '/acs/acs1'
    g_ = '?get='
    
    # User inputs
    api_key_ = f"&key={api_key}"
    variables_ = variables
    year_ = '/' + str(year)
    location_ = '&for=county:' + county + '&in=state:' + state
    
    # create url query
    query = f"{host_}{year_}{dataset_}{g_}{variables_}{location_}{api_key_}"
    
    # use requests package to call out to the API
    response = requests.get(query).text
    response = response.replace('null', '"null"')
    response = ast.literal_eval(response)
    
    # convert parsed response text to pandas df
    df_acs = pd.DataFrame(response[1:], columns = response[0])
    
    # apply year tag
    df_acs['Year'] = year
    
    return df_acs



#### File paths

In [ ]:
# Working directory
path_projects = os.path.dirname(os.path.dirname(os.getcwd()))
print(path_projects)

# Set file paths
rootpath = os.path.join(path_projects, 'SACOG Exam')
path_in     = os.path.join(rootpath, 'Raw Data'     )
path_out    = os.path.join(rootpath, 'Python Output')
path_config = os.path.join(rootpath, 'config'       )

#### Set API key

In [ ]:
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

## Prepare Mapping Tables

#### Import FIPS mapping table

In [2]:
# Use URL to county fips mapping table
# Import county FIPS codes by state
url_fips = "https://www2.census.gov/geo/docs/reference/codes/files/national_county.txt"
df_fips = pd.read_csv(url_fips, header = None, sep = ',')
df_fips.head()

,0,1,2,3,4
0,AL,1,1,Autauga County,H1
1,AL,1,3,Baldwin County,H1
2,AL,1,5,Barbour County,H1
3,AL,1,7,Bibb County,H1
4,AL,1,9,Blount County,H1


In [3]:
# Clean and subset FIPS file
# rename columns
# clean county name
# reformat FIPS field

df_fips.columns = ['State', 'State FIPS', 'County FIPS', 'County', 'idk what this is tbh']
df_fips['County'] = df_fips['County'].str.replace(' County', '')
df_fips['County FIPS'] = df_fips['County FIPS'].astype(str).apply('{:0>3}'.format)
df_fips['State FIPS' ] = df_fips['State FIPS' ].astype(str).apply('{:0>2}'.format)

# show
df_fips.head()

,State,State FIPS,County FIPS,County,idk what this is tbh
0,AL,01,001,Autauga,H1
1,AL,01,003,Baldwin,H1
2,AL,01,005,Barbour,H1
3,AL,01,007,Bibb,H1
4,AL,01,009,Blount,H1


In [ ]:
# Export locally just in case url gets moved
df_fips.to_excel(os.path.join(path_out, 'County to FIPS Code Mapping.xlsx'), index=False)

#### Prepare transportation variables

In [ ]:
# Import latest variable labels
year = 2022
url_table_vars = f'https://api.census.gov/data/{year}/acs/acs1/variables.html'
table_vars = pd.read_html(url_table_vars)
df_vars = pd.DataFrame(table_vars[0])

# clean label field
df_vars['Label'] = df_vars['Label'].str.replace('Estimate!!', '')
df_vars['Label'] = df_vars['Label'].str.replace('!!', ' ')
df_vars['Label'] = df_vars['Label'].str.replace(':', '')

# show
df_vars.head()

In [ ]:
# Subset to just transportation related tables
mask_trans = df_vars['Concept'].astype(str).str.contains('transportation', regex = True, case = False)
df_vars = df_vars[mask_trans]
df_vars.head()

In [ ]:
# df_vars['Name'].values.tolist()

## Import Data

#### Prepare inputs for importing ACS data

In [ ]:
# Prepare inputs for census data

# Counties to subset
counties = ['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba']
df_fips = df_fips[df_fips['County'].isin(counties)]

# set state, counties, years, and variables to import
state = states.CA.fips
counties_to_import = df_fips['County FIPS'].astype(str).unique().tolist()
years_to_import = list(range(2018, 2023))
list_vars = ['NAME', 
                    'B08006_001E',
                    'B08006_002E',
                    'B08006_003E',
                    'B08006_004E',
                    'B08006_005E',
                    'B08006_006E',
                    'B08006_007E',
                    'B08006_008E',
                    'B08006_009E',
                    'B08006_010E',
                    'B08006_011E',
                    'B08006_012E',
                    'B08006_013E',
                    'B08006_014E',
                    'B08006_015E',
                    'B08006_016E',
                    'B08006_017E']
variables = ",".join(list_vars)

#### Import ACS 1-Year Estimates

In [ ]:
## Import Block ##

# initialize empty list to store data frames
# import multiple years and counties

list_df_acs = []

for year in tqdm(years_to_import):
    for county in counties_to_import:
        try:
            list_df_acs.append(
                acs1_state_county(api_Key = api_key
                                  , variables = variables
                                  , year = year
                                  , state = state
                                  , county = county)
            )
        except:
            pass
        
df_acs = pd.concat(list_df_acs)

In [ ]:
df_acs.head()

## Data Cleaning

In [ ]:
# reorganize column names
column_names = df_vars[df_vars['Name'].isin(list_vars)]['Label'].tolist()
column_names.insert(0, 'NAME')
column_names = column_names + ['state', 'county', 'year']
df_acs.columns = column_names
df_acs.head(3)

In [ ]:
# Check column names
df_acs.columns

In [ ]:
# Manually check column names and clean as needed
df_acs = df_acs[[
    'state', 'county', 'NAME', 'year', 
    'Total',
    'Total Car, truck, or van Drove alone',
    'Total Worked from home'
]].rename(columns = {
    'state':'State FIPS'
    , 'year':'Year'
    , 'NAME':'County Name'
    , 'county':'County FIPS'
    , 'Total':'Population'
    , 'Total Car, truck, or van Drove alone':'Population commuting alone'
    , 'Total Worked from home':'Population working from home'
})

# display
df_acs.head(3)

In [ ]:
# Clean field types and county names
df_acs = df_acs[df_acs != 'null']
df_acs[['Population'
            , 'Population commuting alone'
            , 'Population working from home'
]] = df_acs[['Population'
            , 'Population commuting alone'
            , 'Population working from home'
]].apply(pd.to_numeric)
df_acs['County Name'] = df_acs['County Name'].str.replace(' County, California', '')

In [ ]:
df_acs.head()

## Data Checks

In [ ]:
# Check years and counties
# Check missings
# Check duplicates

# assert df_acs['Year']       .unique().tolist() == years_to_import
assert df_acs['County FIPS'].unique().tolist() == counties_to_import
# assert pd.DataFrame(df_acs.isnull().sum(axis = 0))[0].sum() == 0
assert df_acs[['County FIPS', 'Year']].duplicated().sum() == 0

Notes on assert statements:

- There is no data collected for ACS 1-year estimates in 2020
    - Probably because of covid
- Sutter county and Yuba county do not have any transportation data for all years
    - Maybe because population is too small?
    - https://www.census.gov/programs-surveys/acs/guidance/estimates.html this source claims that it has data for areas with 65k or more people (for 2022).  Yuba and Sutter have had more than 65k people since 2005, so not sure if that's it.

In [ ]:
# Check summaries of quantitative variables
pd.options.display.float_format = "{:.0f}".format
round(df_acs.describe())

## Export

In [ ]:
# # Set output name
# name_output = ['Step 01a_Cleaned ACS1 Table B08006_', date.today().strftime("%Y-%m-%d"),'.xlsx']
# name_output = "".join(name_output)

In [ ]:
# # Export locally just in case url gets moved
# df_acs.to_excel(os.path.join(path_out, name_output), index=False)